# ScanForge backend + GPU worker on Colab

Runs the whole server side for testing: the FastAPI backend, a public ngrok address the phone
app talks to, and the COLMAP reconstruction on Colab's GPU.

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the cells **top to bottom**. The last cell keeps running and processes scans as they arrive.
3. Keep this tab open. When Colab disconnects, scans stop and stored jobs are lost. Rerun all cells to start again.

## 1. Settings

In [ ]:
# Your ngrok token: add it under 🔑 Secrets (left sidebar) as NGROK_TOKEN with notebook access on,
# or paste it when asked. Don't commit the token: this repo is public.
NGROK_DOMAIN = "chamber-barber-dole.ngrok-free.dev"   # must match the APK's API base

import getpass
try:
    from google.colab import userdata
    NGROK_TOKEN = userdata.get("NGROK_TOKEN")
except Exception:
    NGROK_TOKEN = None
if not NGROK_TOKEN:
    NGROK_TOKEN = getpass.getpass("ngrok auth token: ").strip()
print("token loaded")

## 2. Get the code and backend dependencies (~1 min)

In [ ]:
# Uses Colab's existing package versions (no pins), so nothing Colab relies on gets downgraded.
!rm -rf /content/scanforge && git clone -q --depth 1 https://github.com/infinityinketan-sketch/scanforge.git /content/scanforge
!pip -q install fastapi "uvicorn[standard]" python-multipart httpx google-api-python-client google-auth pyngrok

import importlib
for m in ["fastapi", "uvicorn", "multipart", "httpx", "googleapiclient", "pyngrok"]:
    importlib.import_module(m)
import os; assert os.path.exists("/content/scanforge/backend/main.py"), "clone failed"
print("✅ backend ready (a red 'pip dependency resolver' note above is harmless)")


## 3. Install COLMAP with CUDA (~5–10 min, once per session)

COLMAP's dense step needs a CUDA build, so it comes from conda-forge into its own environment
(`/content/recon`), separate from Colab's Python.

In [ ]:
%%bash
set -e
nvidia-smi -L || { echo "No GPU: Runtime > Change runtime type > T4 GPU, then rerun"; exit 1; }
cd /content
if [ ! -x /content/mm/bin/micromamba ]; then
  mkdir -p mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C mm bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/mm
if [ ! -x /content/recon/bin/colmap ]; then
  /content/mm/bin/micromamba create -q -y -p /content/recon -c conda-forge python=3.11 "colmap=*=*cuda*" \
    || { echo "!! No CUDA COLMAP build found; installing CPU build (dense step will fail)";
         /content/mm/bin/micromamba create -q -y -p /content/recon -c conda-forge python=3.11 colmap; }
  /content/recon/bin/python -m pip -q install open3d trimesh requests numpy
fi
/content/recon/bin/colmap -h | head -2

## 4. Start the backend and the public ngrok address

In [ ]:
import os, secrets, subprocess, sys, time, requests
from pyngrok import ngrok

API_BASE = f"https://{NGROK_DOMAIN}"
os.environ["DATA_DIR"] = "/content/data"
os.environ["API_BASE"] = API_BASE
os.environ["ALLOW_DEV_BILLING"] = "1"   # test only: lets the helper below unlock exports
os.environ.setdefault("SECRET_KEY", secrets.token_hex(16))   # stable for this session
SECRET = os.environ["SECRET_KEY"]

# Restart cleanly if this cell is rerun
subprocess.run("pkill -f 'uvicorn main:app'", shell=True)
ngrok.kill()
time.sleep(1)

log_file = open("/content/backend.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/scanforge/backend", env=os.environ.copy(),
    stdout=log_file, stderr=subprocess.STDOUT,
)
for _ in range(30):
    time.sleep(1)
    try:
        if requests.get("http://localhost:8000/healthz", timeout=2).ok:
            break
    except requests.RequestException:
        pass
else:
    print(open("/content/backend.log").read()[-3000:])
    raise RuntimeError("Backend did not start; log above")

ngrok.set_auth_token(NGROK_TOKEN)
ngrok.connect(8000, domain=NGROK_DOMAIN)
r = requests.get(f"{API_BASE}/healthz", headers={"ngrok-skip-browser-warning": "1"}, timeout=15)
print("public health check:", r.status_code, r.text)
print("✅ Backend live at", API_BASE)

## 5. Process scans (leave running)

Picks up every uploaded scan, runs COLMAP + meshing on the GPU, and posts the GLB/STL back.
The phone app updates by itself when a scan finishes. Stop this cell to stop processing.

In [ ]:
import json
sys.path.insert(0, "/content/scanforge/backend")
import db   # same SQLite file as the server (DATA_DIR above)

LOCAL = "http://localhost:8000"
RUNNER = "import sys, json; sys.path.insert(0, '/content/scanforge/pipeline'); import handler; " \
         "print(json.dumps(handler.handler(json.loads(sys.argv[1]))['ok']))"
env = dict(os.environ, PATH="/content/recon/bin:" + os.environ["PATH"], JOB_DIR="/content/work")

def run_job(job_id):
    m = requests.get(f"{LOCAL}/jobs/{job_id}/manifest", params={"key": SECRET}, timeout=30).json()
    # Fetch photos and post results over localhost instead of through ngrok.
    swap = lambda u: u.replace(API_BASE, LOCAL, 1)
    event = {"input": {"job_id": job_id,
                       "photo_urls": [swap(u) for u in m["photo_urls"]],
                       "result_url": swap(m["result_url"])}}
    print(f"▶ {job_id}: {len(event['input']['photo_urls'])} photos")
    p = subprocess.run(["/content/recon/bin/python", "-c", RUNNER, json.dumps(event)],
                       env=env, capture_output=True, text=True)
    open(f"/content/worker-{job_id}.log", "w").write(p.stdout + p.stderr)
    job = db.get_job(job_id)
    if job["status"] == "processing":   # worker crashed before reporting back
        tail = (p.stderr or p.stdout).strip().splitlines()[-1:] or ["worker crashed"]
        db.update_job(job_id, status="failed", error=tail[0][:500], finished=time.time())
    print(f"  {job_id}: {db.get_job(job_id)['status']}  {db.get_job(job_id)['error'] or ''}")

print("Waiting for scans…")
while True:
    for row in db._conn().execute("SELECT id FROM jobs WHERE status='queued' ORDER BY created_at").fetchall():
        if db.claim_status(row["id"], ("queued",), "processing", started=time.time()):
            try:
                run_job(row["id"])
            except Exception as e:
                db.update_job(row["id"], status="failed", error=f"worker: {e}"[:500], finished=time.time())
                print("  failed:", e)
    time.sleep(5)

## Testing helpers (stop step 5 first, run, then restart step 5)

The APK is sideloaded, so Google Play can't sell the export unlock. This marks the most recent finished scan as paid
so you can test GLB/STL downloads.

In [ ]:
row = db._conn().execute("SELECT id FROM jobs WHERE status='done' ORDER BY finished DESC LIMIT 1").fetchone()
if row:
    print(requests.post(f"{LOCAL}/jobs/{row['id']}/dev-pay", params={"key": SECRET}).json())
else:
    print("no finished scans yet")

In [ ]:
# Recent jobs and the backend log
for r in db._conn().execute("SELECT id, status, n_photos, error FROM jobs ORDER BY created_at DESC LIMIT 10"):
    print(dict(r))
print(open("/content/backend.log").read()[-2000:])